# D01 — 삼성전자 corp_code 조회

OpenDART 기업고유번호 API에서 종목코드 `005930`에 해당하는 기업을 찾습니다. 프로젝트 폴더의 `.env`에서 `DART_API_KEY`를 읽으며, 키를 출력하거나 노트북에 저장하지 않습니다.

실행 전 `.env`에 `DART_API_KEY=발급받은키`가 있는지 확인하세요. 이 노트북은 API 호출 1회로 전체 기업코드 ZIP 목록을 받고, 메모리에서 삼성전자 행만 골라 출력합니다.

In [11]:
from pathlib import Path
import os
import urllib.parse
import urllib.request
import zipfile
import io
import xml.etree.ElementTree as ET

In [12]:
def find_env_file(start: Path) -> Path | None:
    for folder in (start, *start.parents):
        candidate = folder / '.env'
        if candidate.is_file():
            return candidate
    return None

def read_env_value(env_path: Path, name: str) -> str | None:
    for raw_line in env_path.read_text(encoding='utf-8-sig').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        if key.strip() == name:
            return value.strip().strip('\"').strip("'")
    return None

In [16]:
env_path = find_env_file(Path.cwd())
api_key = os.environ.get('DART_API_KEY')
if not api_key and env_path:
    api_key = read_env_value(env_path, 'DART_API_KEY')
if not api_key:
    raise RuntimeError('DART_API_KEY를 환경변수 또는 프로젝트 .env에 설정하세요.')

url = 'https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key=' + urllib.parse.quote(api_key, safe='')
request = urllib.request.Request(url, headers={'User-Agent': 'D01-corp-code-check/1.0'}, method="GET")
try:
    with urllib.request.urlopen(request, timeout=60) as response:
        http_status = response.status
        payload = response.read()
except Exception as exc:
    # 예외 문자열에는 인증키가 포함된 요청 URL이 들어갈 수 있어 출력하지 않습니다.
    raise RuntimeError(f'OpenDART 요청 실패 ({type(exc).__name__}). 네트워크와 인증키 상태를 확인하세요.') from None

with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    xml_names = [name for name in archive.namelist() if name.lower().endswith('.xml')]
    if not xml_names:
        raise RuntimeError('응답 ZIP 안에서 XML 파일을 찾지 못했습니다.')
    root = ET.fromstring(archive.read(xml_names[0]))

print(f'HTTP {http_status}')  # 정상 응답이면 보통 HTTP 200

def field_text(element, field):
    child = element.find(field)
    return child.text.strip() if child is not None and child.text else ''

HTTP 200


In [14]:
import json

def save_company_info_json(stock_code: str, output_dir: str | Path = 'dart_results') -> tuple[dict, Path]:
    """종목코드에 해당하는 기업 정보를 JSON 파일로 저장하고 (데이터, 파일 경로)를 반환합니다."""
    code = str(stock_code).strip()
    if not code.isdigit() or len(code) > 6:
        raise ValueError('종목코드는 숫자 1~6자리로 입력하세요.')
    code = code.zfill(6)  # 선행 0 보존

    matches = [item for item in root.iter('list') if field_text(item, 'stock_code') == code]
    if not matches:
        status = root.findtext('.//status') or ''
        message = root.findtext('.//message') or ''
        if status or message:
            raise RuntimeError(f'OpenDART 응답 오류: status={status}, message={message}')
        raise LookupError(f'종목코드 {code}에 해당하는 기업을 찾지 못했습니다.')
    if len(matches) > 1:
        raise RuntimeError(f'종목코드 {code} 결과가 {len(matches)}건입니다.')

    item = matches[0]
    returned_code = field_text(item, 'stock_code')
    info = {
        'corp_name': field_text(item, 'corp_name'),
        'stock_code': returned_code or None,
        'corp_code': field_text(item, 'corp_code'),
        'modify_date': field_text(item, 'modify_date'),
        'listing_status': '상장' if returned_code else '비상장',
    }

    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    json_path = destination / f'company_stock_{code}.json'
    json_path.write_text(json.dumps(info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return info, json_path

company_info, json_path = save_company_info_json(input('종목코드를 입력하세요: '))
print(json.dumps(company_info, ensure_ascii=False, indent=2))
print(f'JSON 파일 저장 위치: {json_path.resolve()}')

종목코드를 입력하세요:  005930


{
  "corp_name": "삼성전자",
  "stock_code": "005930",
  "corp_code": "00126380",
  "modify_date": "20251201",
  "listing_status": "상장"
}
JSON 파일 저장 위치: C:\ai_campus\proj\dart_results\company_005930.json


## 검증로그에 기록할 내용

함수 `save_company_info_json(종목코드)`는 `dart_results/company_<종목코드>.json`에 `corp_name`, `stock_code`, `corp_code`, `modify_date`, `listing_status`를 저장하고 `(정보 dict, 파일 경로)`를 반환합니다.

OpenDART 기업코드 목록은 비상장사의 `stock_code`가 비어 있을 수 있습니다. 아래 `save_company_info_by_corp_code_json(corp_code)` 함수는 `corp_code`로 기업을 찾아 `stock_code`가 있으면 상장, 비어 있으면 비상장으로 저장합니다. 이 목록에 없는 기업은 비상장으로 간주하지 않고 조회 오류로 처리합니다. 인증키는 로그·출력·공유 파일에 복사하지 마세요.

In [19]:
def save_company_info_by_corp_code_json(corp_code: str, output_dir: str | Path = 'dart_results') -> tuple[dict, Path]:
    """DART corp_code로 기업을 찾아 상장 여부를 판별하고 JSON 파일과 (정보, 경로)를 반환합니다."""
    code = str(corp_code).strip()
    if not code.isdigit() or len(code) != 8:
        raise ValueError('corp_code는 8자리 숫자로 입력하세요.')

    matches = [item for item in root.iter('list') if field_text(item, 'corp_code') == code]
    if not matches:
        status = root.findtext('.//status') or ''
        message = root.findtext('.//message') or ''
        if status or message:
            raise RuntimeError(f'OpenDART 응답 오류: status={status}, message={message}')
        raise LookupError(f'corp_code {code}를 기업코드 목록에서 찾지 못했습니다.')
    if len(matches) > 1:
        raise RuntimeError(f'corp_code {code} 결과가 {len(matches)}건입니다.')

    item = matches[0]
    stock_code = field_text(item, 'stock_code')
    info = {
        'corp_name': field_text(item, 'corp_name'),
        'stock_code': stock_code or None,
        'corp_code': field_text(item, 'corp_code'),
        'modify_date': field_text(item, 'modify_date'),
        'listing_status': '상장' if stock_code else '비상장',
    }

    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    json_path = destination / f'company_corp_{code}.json'
    json_path.write_text(json.dumps(info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return info, json_path

corp_info, corp_json_path = save_company_info_by_corp_code_json(
    input('8자리 corp_code를 입력하세요: ')
)
print(json.dumps(corp_info, ensure_ascii=False, indent=2))
print(f'JSON 파일 저장 위치: {corp_json_path.resolve()}')

8자리 corp_code를 입력하세요:  00252074


{
  "corp_name": "삼성전자판매",
  "stock_code": null,
  "corp_code": "00252074",
  "modify_date": "20240124",
  "listing_status": "비상장"
}
JSON 파일 저장 위치: C:\ai_campus\proj\dart_results\company_corp_00252074.json
